# EDA — Emerging Market Data

Comprehensive exploratory data analysis for the emerging-market processed data (Part 2).

Produces per-line (motor, buildings, contents):
- **Value counts** for all categorical variables
- **Quantiles** for all numeric variables
- **Outlier flags** using the IQR method (1.5 × IQR beyond Q1/Q3)
- **Boxplots** for each numeric variable (seaborn)
- **Histograms** for each numeric variable (to reveal distribution shape / skew)
- **Correlation matrix** heatmap

All outputs are saved to an Excel workbook per line in `Excel files_emerging_v2/`.

In [0]:
# Step 1: Load processed emerging data and classify columns
%pip install seaborn openpyxl -q

import pandas as pd
import numpy as np
import os, io
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data_emerging/'
PROC_DIR = os.path.join(DATA_DIR, 'proc')
EXCEL_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/Excel files_emerging_v2/'
os.makedirs(EXCEL_DIR, exist_ok=True)

# Columns to exclude from EDA (IDs, dates, not meaningful as features)
EXCLUDE_COLS = {
    'row_id', 'policy_id', 'customer_id', 'address_id',
    'period_start', 'period_end', 'policyholder_id'
}

# Load all three lines
train = {}
cat_cols = {}
num_cols = {}

for line in ['motor', 'buildings', 'contents']:
    train[line] = pd.read_parquet(os.path.join(PROC_DIR, f'proc_{line}.parquet'))
    print(f'\n{line.upper()}: {train[line].shape}')

    # Identify categorical and numeric columns (excluding IDs)
    cats = []
    nums = []
    for col in train[line].columns:
        if col in EXCLUDE_COLS:
            continue
        dtype = train[line][col].dtype
        n_unique = train[line][col].nunique()
        is_cat = (dtype == 'object' or dtype == 'bool')

        if is_cat:
            cats.append(col)
        elif pd.api.types.is_numeric_dtype(train[line][col]):
            # Treat low-cardinality integers (<=15 unique) like Year as categorical
            if n_unique <= 15 and 'Year' in col:
                cats.append(col)
            else:
                nums.append(col)
        else:
            cats.append(col)

    cat_cols[line] = cats
    num_cols[line] = nums
    print(f'  Categorical ({len(cats)}): {cats}')
    print(f'  Numeric ({len(nums)}): {nums}')

print('\nData loaded and classified.')

In [0]:
# Step 2: Value counts for categorical variables -> Excel
from openpyxl import Workbook

for line in ['motor', 'buildings', 'contents']:
    df = train[line]
    cats = cat_cols[line]
    excel_path = os.path.join(EXCEL_DIR, f'EDA_{line}.xlsx')

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # --- Value counts for each categorical variable ---
        for col in cats:
            vc = df[col].value_counts(dropna=False)
            sheet_name = f'vc_{col}'[:31]
            vc.rename('count').rename_axis(col).to_excel(writer, sheet_name=sheet_name)

    print(f'{line}: saved {len(cats)} value-count sheets to {excel_path}')

print('\nStep 2 complete: categorical value counts saved.')

In [0]:
# Step 3: Quantiles + outlier detection for numeric variables -> Excel
from openpyxl import load_workbook

for line in ['motor', 'buildings', 'contents']:
    df = train[line]
    nums = num_cols[line]
    excel_path = os.path.join(EXCEL_DIR, f'EDA_{line}.xlsx')

    # --- Quantiles table ---
    quantile_labels = [0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
    quantile_rows = []
    outlier_rows = []

    for col in nums:
        s = df[col].dropna()
        if len(s) == 0:
            continue
        q = s.quantile(quantile_labels)
        q_dict = {'variable': col}
        q_dict.update({f'q{int(q_label*100) if q_label != 1.0 else 100}': q_val for q_label, q_val in zip(quantile_labels, q)})
        quantile_rows.append(q_dict)

        # IQR outlier detection: values beyond Q1 - 1.5*IQR or Q3 + 1.5*IQR
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        n_low = (s < lower).sum()
        n_high = (s > upper).sum()
        n_total_outliers = n_low + n_high
        pct_outliers = n_total_outliers / len(s) * 100
        outlier_rows.append({
            'variable': col,
            'Q1': q1, 'Q3': q3, 'IQR': iqr,
            'lower_fence': lower, 'upper_fence': upper,
            'n_below': n_low, 'n_above': n_high,
            'n_outliers': n_total_outliers,
            'pct_outliers': round(pct_outliers, 2),
            'flag': '*** OUTLIER ***' if pct_outliers > 5 else ('** borderline **' if pct_outliers > 1 else '')
        })

    quantile_df = pd.DataFrame(quantile_rows)
    outlier_df = pd.DataFrame(outlier_rows)

    # Append to existing workbook
    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        quantile_df.to_excel(writer, sheet_name='quantiles', index=False)
        outlier_df.to_excel(writer, sheet_name='outlier_flags', index=False)

    # Print outliers flagged as >5%
    flagged = outlier_df[outlier_df['pct_outliers'] > 5].sort_values('pct_outliers', ascending=False)
    print(f'\n{line.upper()} — {len(flagged)} variables with >5% outliers:')
    for _, r in flagged.iterrows():
        print(f'  {r["variable"]:30s} {r["pct_outliers"]:6.1f}%  ({r["n_outliers"]} obs, fence=[{r["lower_fence"]:.2f}, {r["upper_fence"]:.2f}])')

print('\nStep 3 complete: quantiles and outlier flags saved.')

In [0]:
# Step 4: Boxplots for numeric variables -> embed as images in Excel
from openpyxl.drawing.image import Image as XLImage

for line in ['motor', 'buildings', 'contents']:
    df = train[line]
    nums = num_cols[line]
    excel_path = os.path.join(EXCEL_DIR, f'EDA_{line}.xlsx')
    wb = load_workbook(excel_path)

    for col in nums:
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.boxplot(data=df, y=col, ax=ax)
        ax.set_title(f'{line} — {col}')
        ax.set_ylabel(col)
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = XLImage(buf)
        sheet_name = f'{col}_box'[:31]
        ws = wb.create_sheet(sheet_name)
        ws.add_image(img, 'B2')

    wb.save(excel_path)
    print(f'{line}: saved {len(nums)} boxplots')

print('\nStep 4 complete: boxplots saved to Excel.')

In [0]:
# Step 5: Histograms for numeric variables -> embed as images in Excel
# Histograms reveal distribution shape (skew, multimodality) that boxplots miss

for line in ['motor', 'buildings', 'contents']:
    df = train[line]
    nums = num_cols[line]
    excel_path = os.path.join(EXCEL_DIR, f'EDA_{line}.xlsx')
    wb = load_workbook(excel_path)

    for col in nums:
        s = df[col].dropna()
        if len(s) == 0:
            continue
        fig, ax = plt.subplots(figsize=(6, 4))
        # Use log scale for heavily skewed variables (ultimate_claims, written_premium, sum_insured)
        if col in ('ultimate_claims', 'written_premium', 'sum_insured', 'reported_incurred') and s.max() > 1000:
            pos_s = s[s > 0]
            if len(pos_s) > 10:
                ax.hist(pos_s, bins=50, color='steelblue', edgecolor='white')
                ax.set_xscale('log')
                ax.set_title(f'{line} — {col} (log x-axis, zeros excluded)')
            else:
                ax.hist(s, bins=50, color='steelblue', edgecolor='white')
                ax.set_title(f'{line} — {col}')
        else:
            ax.hist(s, bins=50, color='steelblue', edgecolor='white')
            ax.set_title(f'{line} — {col}')
        ax.set_xlabel(col)
        ax.set_ylabel('count')

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = XLImage(buf)
        sheet_name = f'{col}_hist'[:31]
        ws = wb.create_sheet(sheet_name)
        ws.add_image(img, 'B2')

    wb.save(excel_path)
    print(f'{line}: saved {len(nums)} histograms')

print('\nStep 5 complete: histograms saved to Excel.')

In [0]:
# Step 6: Correlation matrices -> embed heatmap images + data in Excel

for line in ['motor', 'buildings', 'contents']:
    df = train[line]
    nums = num_cols[line]
    excel_path = os.path.join(EXCEL_DIR, f'EDA_{line}.xlsx')

    # Compute correlation matrix
    corr_df = df[nums].corr()

    # Save correlation data as a sheet
    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        corr_df.to_excel(writer, sheet_name='correlation_matrix')

    # Save heatmap image
    wb = load_workbook(excel_path)
    fig, ax = plt.subplots(figsize=(max(10, len(nums) * 0.6), max(8, len(nums) * 0.5)))
    sns.heatmap(corr_df, annot=True, cmap='coolwarm', fmt='.2f', 
                center=0, square=True, ax=ax, 
                annot_kws={'size': 6 if len(nums) > 15 else 8})
    ax.set_title(f'{line.upper()} — Correlation Matrix')
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    img = XLImage(buf)
    ws = wb.create_sheet('corr_heatmap')
    ws.add_image(img, 'B2')

    wb.save(excel_path)

    # Print top correlations with ultimate_claims
    if 'ultimate_claims' in corr_df.columns:
        top_corr = corr_df['ultimate_claims'].drop('ultimate_claims').sort_values(key=abs, ascending=False)
        print(f'\n{line.upper()} — Top correlations with ultimate_claims:')
        for var, val in top_corr.head(10).items():
            print(f'  {var:35s} {val:+.4f}')

print('\nStep 6 complete: correlation matrices saved.')

In [0]:
# Step 7: Summary — print sheet counts and file info
print('='*70)
print('EDA SUMMARY — Emerging Market Data')
print('='*70)

for line in ['motor', 'buildings', 'contents']:
    excel_path = os.path.join(EXCEL_DIR, f'EDA_{line}.xlsx')
    wb = load_workbook(excel_path)
    sheet_names = wb.sheetnames

    vc_sheets = [s for s in sheet_names if s.startswith('vc_')]
    box_sheets = [s for s in sheet_names if s.endswith('_box')]
    hist_sheets = [s for s in sheet_names if s.endswith('_hist')]

    print(f'\n{line.upper()}: {os.path.getsize(excel_path) / 1e6:.1f} MB')
    print(f'  Total sheets: {len(sheet_names)}')
    print(f'  Value count sheets: {len(vc_sheets)}')
    print(f'  Boxplot sheets: {len(box_sheets)}')
    print(f'  Histogram sheets: {len(hist_sheets)}')
    print(f'  Quantiles sheet: {"quantiles" in sheet_names}')
    print(f'  Outlier flags sheet: {"outlier_flags" in sheet_names}')
    print(f'  Correlation matrix sheet: {"correlation_matrix" in sheet_names}')
    print(f'  Correlation heatmap sheet: {"corr_heatmap" in sheet_names}')
    print(f'  File: {excel_path}')

print('\nAll EDA outputs saved to:', EXCEL_DIR)